# Week 8 — Evaluation Expansion

This notebook evaluates the final XGBoost model using additional
percentage-based metrics and examines performance across property-price
bands.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

y_test = np.load("week7_y_test.npy")
xgb_predictions = np.load("week7_xgb_predictions.npy")

print("Actual prices:", y_test.shape)
print("Predictions:", xgb_predictions.shape)

assert len(y_test) == len(xgb_predictions)

Actual prices: (11914,)
Predictions: (11914,)


Absolute percentage error =
|actual price - predicted price| / actual price × 100

MAPE - the mean percentage error accorss all properties 

MdAPE - the median percentage error

In [7]:
absolute_percentage_errors = (
    np.abs(y_test - xgb_predictions)
    / y_test
) * 100

print(
    "First 10 percentage errors:",
    absolute_percentage_errors[:10]
)

First 10 percentage errors: [ 0.61184322 14.16942857  3.9502877  19.39344048  7.02869868  1.03298438
  8.62794258  7.49043321 10.41928888  3.07249531]


In [8]:
r2 = r2_score(y_test, xgb_predictions)

mae = mean_absolute_error(
    y_test,
    xgb_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        xgb_predictions
    )
)

mape = absolute_percentage_errors.mean()

mdape = np.median(
    absolute_percentage_errors
)

print("R²:", r2)
print("MAE:", mae)
print("RMSE:", rmse)
print("MAPE:", mape, "%")
print("MdAPE:", mdape, "%")

R²: 0.8894820885810153
MAE: 169190.1746146865
RMSE: 301126.91809754726
MAPE: 17.29675060033019 %
MdAPE: 9.813317400830257 %


In [9]:
evaluation_df = pd.DataFrame({
    "ActualPrice": y_test,
    "PredictedPrice": xgb_predictions
})

evaluation_df["AbsoluteError"] = np.abs(
    evaluation_df["ActualPrice"]
    - evaluation_df["PredictedPrice"]
)

evaluation_df["AbsolutePercentageError"] = (
    evaluation_df["AbsoluteError"]
    / evaluation_df["ActualPrice"]
) * 100

evaluation_df.head()

,ActualPrice,PredictedPrice,AbsoluteError,AbsolutePercentageError
0,885000.0,8.795852e+05,5414.81250,0.611843
1,525000.0,5.993895e+05,74389.50000,14.169429
2,315000.0,3.025566e+05,12443.40625,3.950288
3,2100000.0,1.692738e+06,407262.25000,19.393440
4,645500.0,6.001298e+05,45370.25000,7.028699


In [10]:
price_edges = [
    0,
    500_000,
    750_000,
    1_000_000,
    1_500_000,
    2_000_000,
    np.inf
]

price_labels = [
    "Under $500K",
    "$500K-$750K",
    "$750K-$1M",
    "$1M-$1.5M",
    "$1.5M-$2M",
    "$2M+"
]

evaluation_df["PriceBand"] = pd.cut( # reads one ActualPrice at a time and add to each boundary
    evaluation_df["ActualPrice"],
    bins=price_edges,
    labels=price_labels,
    include_lowest=True,
    right=False
)

### **Calculate Overall Metrics**

In [3]:
def calculate_regression_metrics(actual, predicted):
    actual = np.array(actual)
    predicted = np.array(predicted)

    absolute_percentage_errors = (
        np.abs(actual- predicted) / actual
    ) * 100

    return {
        "R2": r2_score(actual, predicted),
        "MAE" : mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAPE": absolute_percentage_errors.mean(),
        "MdAPE": np.median(absolute_percentage_errors)
    }

In [12]:
band_results = []

for price_band, group in evaluation_df.groupby(
    "PriceBand",
    observed=True # Include only price bands that actually contain propertis
):
    metrics = calculate_regression_metrics(
        group["ActualPrice"],
        group["PredictedPrice"]
    )
    band_results.append({
        "Model": "XGBoost",
        "Segment": str(price_band),
        "NumberOfProperties" : len(group),
        **metrics
    })

price_band_metrics = pd.DataFrame(
    band_results
)

price_band_metrics

,Model,Segment,NumberOfProperties,R2,MAE,RMSE,MAPE,MdAPE
0,XGBoost,Under $500K,1644,-1.895458,75393.630157,150717.913291,45.406375,12.171481
1,XGBoost,$500K-$750K,2511,-1.676981,76292.496802,116461.309681,12.294094,8.063960
2,XGBoost,$750K-$1M,2370,-2.819916,96946.898101,138271.879414,11.265448,8.308086
3,XGBoost,$1M-$1.5M,2465,-1.370714,156144.099407,214688.363011,12.698603,9.956771
4,XGBoost,$1.5M-$2M,1315,-4.263337,234158.039068,321294.839823,13.659891,10.880622
5,XGBoost,$2M+,1609,0.535696,483304.523306,666266.929341,15.283413,12.924576


In [14]:
overall_metrics = calculate_regression_metrics(
    y_test,
    xgb_predictions
)

overall_row = pd.DataFrame([{
    "Model": "XGBoost",
    "Segment": "Overall",
    "NumberOfProperties": len(y_test),
    **overall_metrics
}])
overall_row

,Model,Segment,NumberOfProperties,R2,MAE,RMSE,MAPE,MdAPE
0,XGBoost,Overall,11914,0.889482,169190.174615,301126.918098,17.296751,9.813317


In [15]:
metrics_summary = pd.concat(
    [overall_row, price_band_metrics],
    ignore_index=True
)

metrics_summary

,Model,Segment,NumberOfProperties,R2,MAE,RMSE,MAPE,MdAPE
0,XGBoost,Overall,11914,0.889482,169190.174615,301126.918098,17.296751,9.813317
1,XGBoost,Under $500K,1644,-1.895458,75393.630157,150717.913291,45.406375,12.171481
2,XGBoost,$500K-$750K,2511,-1.676981,76292.496802,116461.309681,12.294094,8.063960
3,XGBoost,$750K-$1M,2370,-2.819916,96946.898101,138271.879414,11.265448,8.308086
4,XGBoost,$1M-$1.5M,2465,-1.370714,156144.099407,214688.363011,12.698603,9.956771
5,XGBoost,$1.5M-$2M,1315,-4.263337,234158.039068,321294.839823,13.659891,10.880622
6,XGBoost,$2M+,1609,0.535696,483304.523306,666266.929341,15.283413,12.924576


In [16]:
metrics_summary.to_csv(
    "metrics_summary.csv",
    index=False
)